# Chapter 5 · Brownian Bridges for ABPs — Constructing the Bridge

Sections 5.2–5.4 (pp. 84–96), Figures 32–35.

| Figure | Page | Content |
|---|---|---|
| Figure 32 | p. 86 | Ballistic Brownian motion at different noise levels |
| Figure 33 | p. 91 | Conditioning of a multivariate normal variable |
| Figure 34 | p. 93 | Construction of the conditioned noise |
| Figure 35 | p. 96 | Conditioned vs. unconditioned bridge |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

import lambert_thesis as lt
from lambert_thesis import bridges, plotting
from lambert_thesis.plotting import savefig

plotting.setup()
rng = lt.set_seed(lt.SEED)

## 5.2 Ballistic Brownian motion

The standard ABP moves with $\dot x=v_0\cos\phi+\sqrt{2D_T}\zeta_x$, $\dot y=v_0\sin\phi+\sqrt{2D_T}\zeta_y$,
$\dot\phi=\sqrt{2D_R}\zeta_\phi$. We focus on the rotational diffusion ($D_T=0$) and rescale space and time with
$v_0=1$ and $t^*=1/2D_R$:

$$\dot x=\cos\phi,\qquad \dot y=\sin\phi,\qquad \dot\phi=\sqrt\sigma\,\zeta_\phi .$$

$\sigma=T/t^*$ is the rotational noise accumulated while travelling to the next observation at $t=1$. Less diffusion or
more frequent sampling make the transition less noisy. The swarm of a particle filter spreads perpendicular to the
initial direction in a superdiffusive fashion — the direct reason for the loss in particle-filter performance.

*Code:* `bridges.ballistic_brownian(sigma, n, dt, rng)` integrates the equations with Euler–Maruyama from $t=0$ to $t=1$
(200 particles each, all noise levels driven by the same random numbers; $\Delta t=10^{-2}$, and $\Delta t=10^{-4}$ for the limit $\sigma\to\infty$, represented by $\sigma=200$).

In [ ]:
cases = [  # (sigma, dt, colour, line width, label)
    (200, 1e-4, "grey", 0.8, r"$\sigma=\infty$"),
    (2, 1e-2, "k", 0.8, r"$\sigma=2$"),
    (0.05, 1e-2, "r", 0.7, r"$\sigma=0.05$"),
    (0.01, 1e-2, "b", 0.4, r"$\sigma=0.01$"),
    (0.0, 1e-2, "g", 2.0, r"$\sigma=0.0$"),
]
plt.figure()
for sigma, dt, col, lw, label in cases:
    x, y = bridges.ballistic_brownian(sigma, 200, dt, np.random.default_rng(lt.SEED))   # same noise for every sigma
    plt.plot(x.T, y.T, color=col, lw=lw)
    plt.plot([], [], color=col, lw=lw if lw > 1 else 1, label=label)
plt.plot([0, 1], [0, 0], "o", ms=14, mfc="#1199dd", mec="#114466", mew=2, label="start and next observation")
plt.gca().axis("equal")
plt.legend(loc="upper right", fontsize=8)
plt.grid()
savefig("fig32_noise_levels");

**Figure 32 (p. 86).** Ballistic Brownian motion: rotational but no translational diffusion. Integration from $t=0$ to
$t=1$, where the next observation is assumed (blue circles). High noise curls the trajectories up; without noise the
motion is purely ballistic.

In the paraxial approximation ($\phi$ small, $x(0)=y(0)=\phi(0)=0$) the dynamics decouple, $\dot x\approx1$,
$\dot y\approx\phi$, and the vertical deviation is a doubly integrated noise,
$y(T)=\sqrt\sigma\int_0^T\int_0^t\zeta_\phi\,\mathrm dt$. Small early perturbations magnify over time, which makes it hard
for a Monte-Carlo sampler to propose small changes of the noise.

## 5.3 Conditioning the paraxial solution

We condition the noise on the endpoint $y(T)$ (**shift**) and on the final deviation rate $\dot y(T)$ (**slope**).
Discretised with $N$ Euler–Maruyama steps, $y(T)\approx\sqrt{\sigma\Delta t}\sum_{k=1}^N\sum_{i=1}^kW_i$ with
$W_i\sim\mathcal N(0,1)$. With the cumulative-sum matrix $S_{ij}=[i\le j]$ the zeroth, first and second-order processes
$\mathbf B^{(n)}$ are Gaussian with covariances $\mathbf\Sigma^{(n)}=\mathbf S\mathbf\Sigma^{(n-1)}\mathbf S^T$,
$\mathbf\Sigma^{(0)}=\mathbf I_N$.

Conditioning a Gaussian on part of its components — the last one here — gives
$\bar{\mathbf\mu}=\mathbf\mu_1+\mathbf\Sigma_{12}\mathbf\Sigma_{22}^{-1}(\mathbf a-\mathbf\mu_2)$ and
$\bar{\mathbf\Sigma}=\mathbf\Sigma_{11}-\mathbf\Sigma_{12}\mathbf\Sigma_{22}^{-1}\mathbf\Sigma_{21}$ (Figure 33).
The conditioned covariance does not depend on $\mathbf a$ and can be precomputed.

*Code (Figure 33):* 30 000 samples of a 3D standard normal; for the conditioned variable, samples of 3 million draws
are kept within $|x_1-1|<0.01$. Grey dots: projections onto the side walls; colour: distance from the origin.

In [ ]:
def mvn_3d(pts, d, cut=None, size=5, aspect=1.2, s=0.05, alphas=(0.05, 0.05, 0.05)):
    ax = plt.figure().add_subplot(projection="3d")
    ax.scatter(*pts.T, s=s, c=d, marker=",")
    ax.plot(*(pts * [0, 1, 1] + [-size, 0, 0]).T, ".k", markersize=0.1, alpha=alphas[0])
    ax.plot(*(pts * [1, 0, 1] + [0, size, 0]).T, ".k", markersize=0.1, alpha=alphas[1])
    ax.plot(*(pts * [1, 1, 0] + [0, 0, -size]).T, ".k", markersize=0.1, alpha=alphas[2])
    if cut is not None:
        ax.plot([cut] * 5, [-size, size, size, -size, -size], [-size, -size, size, size, -size], "r", lw=1,
                label=f"conditioning plane $x_1={cut}$")
        ax.legend(loc="upper left", fontsize=8)
    ax.set(xlim=(-size, size), ylim=(-size, size), zlim=(-size / aspect, size / aspect))
    return ax

pts = rng.multivariate_normal(np.zeros(3), np.eye(3), 30_000)
mvn_3d(pts, np.linalg.norm(pts, axis=1))
savefig("fig33a_mvn_samples");

In [ ]:
pts = rng.multivariate_normal(np.zeros(3), np.eye(3), 3_000_000)
d = np.linalg.norm(pts, axis=1)
keep = (pts[:, 0] - 1) ** 2 < 0.01**2
mvn_3d(pts[keep], d[keep] ** (1 / 1.4), cut=1, s=0.1, alphas=(0.1, 0.01, 0.05))
savefig("fig33b_mvn_conditioned");

**Figure 33 (p. 91).** Left: samples from a 3D normal variable. Right: conditioning selects a hyperplane (red); its
intersection with the original distribution is the conditioned variable.

**Algorithm.** (1) Start with $\mathbf\Sigma^{(0)}$; (2) integrate to $\mathbf\Sigma^{(1)}$ (vertical velocity);
(3) condition the final velocity; (4) integrate to $\mathbf\Sigma_v^{(2)}$ (vertical locations); (5) condition the last
location; (6) draw a sample and take its second finite difference to obtain the conditioned noise.

*Code:* `bridges.ConditionedBridge().get(L, endpoint, enddiff, block, corr, rng)` implements steps 1–6; the noise
comes from `bridges.CorrelatedRNG`, which hands out named blocks of Gaussian numbers. Here: 150 bridges of length 500,
conditioned to the endpoint 1000 and the final step 100.

In [ ]:
bridge = bridges.ConditionedBridge()
crng = bridges.CorrelatedRNG(seed=lt.SEED)
endpoint, enddiff = 1000.0, 100.0
B2 = bridge.get(500, endpoint * np.ones(150), enddiff * np.ones(150), "fig34", 0.99, crng)["w"]
B1 = np.concatenate([B2[[0], :], np.diff(B2, axis=0)])
B0 = np.diff(B1, axis=0)

fig, axes = plt.subplots(3, 1, figsize=(5, 10))
for ax, data, label, cond in [(axes[0], B2, r"$\mathbf{B}^{(2)}$", endpoint),
                              (axes[1], B1, r"$\mathbf{B}^{(1)}$", enddiff),
                              (axes[2], B0, r"$\mathbf{B}^{(0)}$", None)]:
    ax.plot(data)
    if cond is not None:
        ax.axhline(cond, label=f"condition: {cond:g}")
        ax.legend(loc="upper center", fontsize=8)
    ax.set_ylabel(label, rotation=0, fontsize=14, labelpad=20)
axes[2].set_xlabel("Index $i$", fontsize=14)
fig.tight_layout()
savefig("fig34_finite_differences");

**Figure 34 (p. 93).** Top: samples of the doubly conditioned process; all end at the same value (blue line). Middle:
the finite differences recover the velocities, also conditioned (blue line). Bottom: the second finite difference is
the noise input that produces these trajectories; it replaces the independent random numbers of the Euler–Maruyama
integration.

## 5.4 Recovering the original distribution

Conditioning biases the noise. To recover the original process, the bridge parameters are marginalised with their
unconditioned joint distribution: with $B^{(2)}_N=\sum_i(N-i+1)W_i$,

$$\mathrm{Var}(B^{(2)}_N)=\frac{N(N+1)(2N+1)}6,\qquad \mathrm{Var}(B^{(1)}_N)=N,\qquad
\mathrm{Cov}(B^{(2)}_N,B^{(1)}_N)=\frac{N(N+1)}2 .$$

*Code:* 5000 bridges of length $L=100$, each with (shift, slope) drawn from this distribution (`bridges.bridge_prior_cov`),
compared with 5000 unconditioned double-cumulative noise sequences.

In [ ]:
L, N = 100, 5000
cov = bridges.bridge_prior_cov(L)
crng = bridges.CorrelatedRNG(seed=lt.SEED)
W = []
for _ in range(N):
    shift, slope = rng.multivariate_normal([0, 0], cov)
    crng.reroll()
    W.append(bridge.get(L, np.array([shift]), np.array([slope]), "fig35", 0, crng)["w"])
W = np.stack(W).squeeze()

ref = rng.standard_normal((N, L))
ref[:, :1] = 0
ref = np.cumsum(np.cumsum(ref, axis=1), axis=1)

### Figure 35 (p. 96), top left: trajectories
The lines alternate every 5 indices between the two processes, otherwise they would lie exactly on top of each other.

In [ ]:
alt_ref, alt_W = ref.copy(), W.copy()
for i in range(L):
    if i % 10 < 5:
        alt_W[:, i] = np.nan
    else:
        alt_ref[:, i] = np.nan
plt.figure()
plt.plot(alt_ref.T, "-b", lw=0.5, alpha=0.25)
plt.plot(alt_W.T, "-r", lw=0.5, alpha=0.25)
plt.plot([], [], "-b", label="unconditioned double-cumulative noise")
plt.plot([], [], "-r", label="bridges, marginalised over (shift, slope)")
plt.xlabel("Index $i$", fontsize=14); plt.ylabel("$B^{(2)}$", fontsize=14)
plt.legend(loc="lower left", fontsize=8)
savefig("fig35a_prior_bridges_trajectories");

### Figure 35, top right: joint distribution of shift and slope

In [ ]:
x = np.linspace(ref[:, -1].min(), ref[:, -1].max(), 500)
y = np.linspace(np.diff(ref, axis=1)[:, -1].min(), np.diff(ref, axis=1)[:, -1].max(), 500)
X, Y = np.meshgrid(x, y)
pdf = multivariate_normal(mean=[0, 0], cov=cov).pdf(np.dstack((X, Y)))

plt.figure()
plt.title("Endpoint vs Enddiff")
plt.plot(ref[:, -1], np.diff(ref, axis=1)[:, -1], ",b")
plt.plot(W[:, -1], np.diff(W, axis=1)[:, -1], ",r")
plt.plot([], [], ".b", label="unconditioned")
plt.plot([], [], ".r", label="bridges")
cs = plt.contour(X, Y, pdf, levels=5, cmap="viridis")
plt.clabel(cs, inline=True, fontsize=8)
plt.plot([], [], color="tab:green", label="density levels of the theoretical distribution")
plt.xlabel("$B_N^{(2)}$", fontsize=16); plt.ylabel("$B_N^{(1)}$", fontsize=16)
plt.legend(loc="upper left", fontsize=8)
savefig("fig35b_prior_bridges_joint");

### Figure 35, bottom: histograms of the final slope and the final shift

In [ ]:
plt.figure()
plt.hist(np.diff(ref, axis=1)[:, -1], bins=20, density=True, histtype="step", label="Reference")
plt.hist(np.diff(W, axis=1)[:, -1], bins=20, density=True, histtype="step", label="Bridge")
s = np.linspace(-5 * np.sqrt(L), 5 * np.sqrt(L), 1000)
plt.plot(s, np.exp(-0.5 * s**2 / L) / np.sqrt(2 * np.pi * L), "g-", lw=2, label="Theoretical")
plt.title("Final velocity $B_N^{(1)}$"); plt.xlabel("Slope"); plt.ylabel("p")
plt.legend()
savefig("fig35c_prior_bridges_slope");

In [ ]:
var2 = (L + 1) * L * (2 * L + 1) / 6
plt.figure()
plt.hist(ref[:, -1], bins=40, density=True, histtype="step", label="Reference")
plt.hist(W[:, -1], bins=40, density=True, histtype="step", label="Bridge")
s = np.linspace(-5 * np.sqrt(var2), 5 * np.sqrt(var2), 1000)
plt.plot(s, np.exp(-0.5 * s**2 / var2) / np.sqrt(2 * np.pi * var2), "g-", lw=2, label="Theoretical")
plt.title("Final Location $B_N^{(2)}$"); plt.xlabel("Shift"); plt.ylabel("p")
plt.legend()
savefig("fig35d_prior_bridges_shift");

**Figure 35 (p. 96).** Conditioned vs. unconditioned bridge. Top left: unconditioned double-cumulative noise (blue) and
conditioned bridges with marginalised bridge parameters (red). Top right: joint space of shift (x-axis) and slope
(y-axis) with the density levels of the theoretical distribution. Bottom: histograms of the final slope and shift of the
reference (blue), the marginalised bridge (orange) and the theory (green) — the bridge is indistinguishable from the
original process.